In [1]:
# STEP 18: Build final KPI summary and churn-driver evidence

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------
# 1. Load prepared datasets
# ---------------------------------------------------

data_path = Path("../../data/cleaned")

customers = pd.read_csv(
    data_path / "customer_features.csv"
)

monthly_usage = pd.read_csv(
    data_path / "monthly_usage_features.csv"
)

monthly_churn = pd.read_csv(
    data_path / "monthly_churn_summary.csv"
)


# ---------------------------------------------------
# 2. Convert date columns
# ---------------------------------------------------

monthly_usage["month"] = pd.to_datetime(
    monthly_usage["month"]
)

monthly_usage["churn_month"] = pd.to_datetime(
    monthly_usage["churn_month"]
)

monthly_churn["month"] = pd.to_datetime(
    monthly_churn["month"]
)


# ---------------------------------------------------
# 3. MAIN BUSINESS KPIs
# ---------------------------------------------------

total_customers = customers["customer_id"].nunique()

total_churned = customers["churn_flag"].sum()

active_customers = (
    total_customers - total_churned
)

overall_churn_rate = (
    total_churned / total_customers * 100
)

average_tenure = customers["tenure_months"].mean()

average_monthly_churn = (
    monthly_churn["monthly_churn_rate_pct"].mean()
)


# ---------------------------------------------------
# 4. First-month engagement KPI
# ---------------------------------------------------

engaged_first_month = customers[
    customers["first_month_workouts"] >= 4
]["customer_id"].nunique()

first_month_engagement_rate = (
    engaged_first_month
    / total_customers
    * 100
)


# ---------------------------------------------------
# 5. Low-engagement churn rate
# ---------------------------------------------------

low_engagement_data = monthly_usage[
    monthly_usage["prev_month_workouts"].notna()
].copy()

low_engagement_data["low_engagement_flag"] = (
    low_engagement_data["prev_month_workouts"] <= 1
).astype(int)


low_engagement_summary = (
    low_engagement_data
    .groupby("low_engagement_flag")
    .agg(
        customer_months=("customer_id", "count"),
        churn_events=("churn_month_flag", "sum")
    )
    .reset_index()
)

low_engagement_summary["churn_rate_pct"] = (
    low_engagement_summary["churn_events"]
    / low_engagement_summary["customer_months"]
    * 100
).round(2)


# ---------------------------------------------------
# 6. Support-contact churn rate
# ---------------------------------------------------

support_summary = (
    monthly_usage
    .groupby("support_contact_flag")
    .agg(
        customer_months=("customer_id", "count"),
        churn_events=("churn_month_flag", "sum")
    )
    .reset_index()
)

support_summary["churn_rate_pct"] = (
    support_summary["churn_events"]
    / support_summary["customer_months"]
    * 100
).round(2)


# ---------------------------------------------------
# 7. Display KPI summary
# ---------------------------------------------------

kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Customers",
        "Active Customers",
        "Churned Customers",
        "Overall Churn Rate (%)",
        "Average Monthly Churn Rate (%)",
        "Average Tenure (Months)",
        "First-Month Engagement Rate (%)"
    ],
    "Value": [
        total_customers,
        active_customers,
        total_churned,
        round(overall_churn_rate, 2),
        round(average_monthly_churn, 2),
        round(average_tenure, 2),
        round(first_month_engagement_rate, 2)
    ]
})

print("FINAL KPI SUMMARY")

display(kpi_summary)


# ---------------------------------------------------
# 8. Display low-engagement evidence
# ---------------------------------------------------

print("\nLOW ENGAGEMENT VS CHURN")

display(low_engagement_summary)


# ---------------------------------------------------
# 9. Display support-contact evidence
# ---------------------------------------------------

print("\nSUPPORT CONTACT VS CHURN")

display(support_summary)

FINAL KPI SUMMARY


,KPI,Value
0,Total Customers,4200.00
1,Active Customers,2432.00
2,Churned Customers,1768.00
3,Overall Churn Rate (%),42.10
4,Average Monthly Churn Rate (%),5.55
5,Average Tenure (Months),6.86
6,First-Month Engagement Rate (%),44.55



LOW ENGAGEMENT VS CHURN


,low_engagement_flag,customer_months,churn_events,churn_rate_pct
0,0,20248,974,4.81
1,1,6077,643,10.58



SUPPORT CONTACT VS CHURN


,support_contact_flag,customer_months,churn_events,churn_rate_pct
0,0,26967,1353,5.02
1,1,3558,415,11.66


In [2]:
# STEP 19: Create final Top 3 churn-driver summary table

# ---------------------------------------------------
# 1. LOW-ENGAGEMENT DRIVER
# ---------------------------------------------------

low_engagement_rate = (
    low_engagement_summary.loc[
        low_engagement_summary["low_engagement_flag"] == 1,
        "churn_rate_pct"
    ]
    .iloc[0]
)

normal_engagement_rate = (
    low_engagement_summary.loc[
        low_engagement_summary["low_engagement_flag"] == 0,
        "churn_rate_pct"
    ]
    .iloc[0]
)


# ---------------------------------------------------
# 2. SUPPORT-CONTACT DRIVER
# ---------------------------------------------------

support_contact_rate = (
    support_summary.loc[
        support_summary["support_contact_flag"] == 1,
        "churn_rate_pct"
    ]
    .iloc[0]
)

no_support_contact_rate = (
    support_summary.loc[
        support_summary["support_contact_flag"] == 0,
        "churn_rate_pct"
    ]
    .iloc[0]
)


# ---------------------------------------------------
# 3. PLAN TYPE DRIVER
# ---------------------------------------------------

plan_type_summary = (
    customers.groupby("plan_type")
    .agg(
        customers=("customer_id", "nunique"),
        churned_customers=("churn_flag", "sum")
    )
    .reset_index()
)

plan_type_summary["churn_rate_pct"] = (
    plan_type_summary["churned_customers"]
    / plan_type_summary["customers"]
    * 100
).round(2)

monthly_plan_rate = (
    plan_type_summary.loc[
        plan_type_summary["plan_type"] == "monthly",
        "churn_rate_pct"
    ]
    .iloc[0]
)

annual_plan_rate = (
    plan_type_summary.loc[
        plan_type_summary["plan_type"] == "annual",
        "churn_rate_pct"
    ]
    .iloc[0]
)


# ---------------------------------------------------
# 4. CREATE FINAL DRIVER TABLE
# ---------------------------------------------------

top_churn_drivers = pd.DataFrame({

    "driver": [
        "Low Previous-Month Engagement",
        "Support Contact",
        "Plan Type / Renewal Risk"
    ],

    "high_risk_group": [
        "0-1 previous-month workouts",
        "Customer contacted support",
        "Monthly plan / annual renewal period"
    ],

    "high_risk_churn_rate_pct": [
        round(low_engagement_rate, 2),
        round(support_contact_rate, 2),
        round(monthly_plan_rate, 2)
    ],

    "comparison_group": [
        "2+ previous-month workouts",
        "No support contact",
        "Annual plan"
    ],

    "comparison_churn_rate_pct": [
        round(normal_engagement_rate, 2),
        round(no_support_contact_rate, 2),
        round(annual_plan_rate, 2)
    ],

    "business_action": [
        "Trigger re-engagement campaigns for inactive members",
        "Prioritize support cases with declining engagement",
        "Use targeted retention and pre-renewal campaigns"
    ]
})


# ---------------------------------------------------
# 5. ADD CHURN RATE DIFFERENCE
# ---------------------------------------------------

top_churn_drivers["churn_rate_gap_pct_points"] = (
    top_churn_drivers["high_risk_churn_rate_pct"]
    -
    top_churn_drivers["comparison_churn_rate_pct"]
).round(2)


# ---------------------------------------------------
# 6. DISPLAY FINAL TABLE
# ---------------------------------------------------

print("TOP 3 CHURN DRIVERS")

display(top_churn_drivers)


# ---------------------------------------------------
# 7. SAVE FOR POWER BI
# ---------------------------------------------------

top_churn_drivers.to_csv(
    "../../outputs/tables/top_churn_drivers.csv",
    index=False
)

kpi_summary.to_csv(
    "../../outputs/tables/kpi_summary.csv",
    index=False
)

low_engagement_summary.to_csv(
    "../../outputs/tables/low_engagement_churn.csv",
    index=False
)

support_summary.to_csv(
    "../../outputs/tables/support_churn.csv",
    index=False
)

print("\nFiles saved successfully for Power BI.")

TOP 3 CHURN DRIVERS


,driver,high_risk_group,high_risk_churn_rate_pct,comparison_group,comparison_churn_rate_pct,business_action,churn_rate_gap_pct_points
0,Low Previous-Month Engagement,0-1 previous-month workouts,10.58,2+ previous-month workouts,4.81,Trigger re-engagement campaigns for inactive m...,5.77
1,Support Contact,Customer contacted support,11.66,No support contact,5.02,Prioritize support cases with declining engage...,6.64
2,Plan Type / Renewal Risk,Monthly plan / annual renewal period,51.18,Annual plan,26.38,Use targeted retention and pre-renewal campaigns,24.80



Files saved successfully for Power BI.


In [3]:
# STEP 20: Final business findings, recommendation, and Power BI outputs

import os

# ---------------------------------------------------
# 1. CREATE OUTPUT FOLDERS IF NEEDED
# ---------------------------------------------------

os.makedirs("../../outputs/tables", exist_ok=True)
os.makedirs("../../outputs/charts", exist_ok=True)
os.makedirs("../../docs", exist_ok=True)


# ===================================================
# 2. ANNUAL RENEWAL ANALYSIS
# ===================================================

# Convert dates if required
customers["signup_date"] = pd.to_datetime(
    customers["signup_date"]
)

monthly_usage["month"] = pd.to_datetime(
    monthly_usage["month"]
)

# Add signup date and plan type to monthly usage
customer_info = customers[
    [
        "customer_id",
        "signup_date",
        "plan_type",
        "cancel_reason"
    ]
].copy()

# Avoid duplicate merge if cell is run again
for col in ["signup_date", "plan_type", "cancel_reason"]:
    if col in monthly_usage.columns:
        monthly_usage = monthly_usage.drop(columns=[col])

monthly_usage = monthly_usage.merge(
    customer_info,
    on="customer_id",
    how="left"
)


# Create signup month
monthly_usage["signup_month"] = (
    monthly_usage["signup_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# Calculate subscription month number
monthly_usage["tenure_month_number"] = (
    (monthly_usage["month"].dt.year -
     monthly_usage["signup_month"].dt.year) * 12
    +
    (
        monthly_usage["month"].dt.month -
        monthly_usage["signup_month"].dt.month
    )
)


# Keep annual-plan customers
annual_usage = monthly_usage[
    monthly_usage["plan_type"] == "annual"
].copy()


# Annual churn by tenure month
annual_renewal_summary = (
    annual_usage
    .groupby("tenure_month_number")
    .agg(
        customer_months=("customer_id", "count"),
        churn_events=("churn_month_flag", "sum")
    )
    .reset_index()
)


annual_renewal_summary["churn_rate_pct"] = (
    annual_renewal_summary["churn_events"]
    /
    annual_renewal_summary["customer_months"]
    * 100
).round(2)


print("ANNUAL PLAN CHURN BY TENURE MONTH")
display(annual_renewal_summary)


# ===================================================
# 3. CANCEL REASON SUMMARY
# ===================================================

cancel_reason_summary = (
    customers[
        customers["churn_flag"] == 1
    ]
    .groupby("cancel_reason")
    .agg(
        churned_customers=("customer_id", "nunique")
    )
    .reset_index()
)


cancel_reason_summary["cancel_reason_pct"] = (
    cancel_reason_summary["churned_customers"]
    /
    cancel_reason_summary["churned_customers"].sum()
    * 100
).round(2)


cancel_reason_summary = (
    cancel_reason_summary
    .sort_values(
        "churned_customers",
        ascending=False
    )
)


print("\nCANCEL REASON SUMMARY")
display(cancel_reason_summary)


# ===================================================
# 4. FIND RENEWAL-PERIOD CHURN
# ===================================================

# Annual subscriptions usually become especially
# important around month 12.

renewal_period = annual_renewal_summary[
    annual_renewal_summary["tenure_month_number"].between(
        11,
        12
    )
]


if len(renewal_period) > 0:

    renewal_churn_rate = (
        renewal_period["churn_events"].sum()
        /
        renewal_period["customer_months"].sum()
        * 100
    )

else:
    renewal_churn_rate = np.nan


# ===================================================
# 5. CREATE FINAL TOP-3 DRIVER TABLE
# ===================================================

final_churn_drivers = pd.DataFrame({

    "driver": [
        "Low Recent Engagement",
        "Support Contact",
        "Annual Renewal Period"
    ],

    "evidence": [
        (
            f"0-1 previous-month workouts: "
            f"{low_engagement_rate:.2f}% churn vs "
            f"{normal_engagement_rate:.2f}% for 2+ workouts"
        ),

        (
            f"Support contact: "
            f"{support_contact_rate:.2f}% churn vs "
            f"{no_support_contact_rate:.2f}% without contact"
        ),

        (
            f"Annual renewal-period churn: "
            f"{renewal_churn_rate:.2f}%"
            if pd.notna(renewal_churn_rate)
            else
            "Annual renewal-period churn analyzed separately"
        )
    ],

    "business_action": [
        "Trigger re-engagement campaigns when recent workout activity falls.",
        "Prioritize support cases for customers whose engagement is declining.",
        "Start renewal campaigns before annual customers reach renewal."
    ],

    "success_metric": [
        "Monthly churn rate and re-engagement rate",
        "Post-support churn and ticket resolution time",
        "Annual renewal rate and renewal-period churn"
    ]
})


print("\nFINAL TOP 3 CHURN DRIVERS")
display(final_churn_drivers)


# ===================================================
# 6. CREATE ONE-PAGE BUSINESS RECOMMENDATION
# ===================================================

recommendation = f"""
AI-POWERED SUBSCRIPTION CHURN & RETENTION INTELLIGENCE SYSTEM
FINAL BUSINESS RECOMMENDATION

BUSINESS PROBLEM
Leadership wants to understand why subscribers cancel and
which actions could improve retention.

KEY FINDINGS

1. LOW RECENT ENGAGEMENT
Customers completing 0-1 workouts in the previous month had
a churn rate of {low_engagement_rate:.2f}%, compared with
{normal_engagement_rate:.2f}% among customers completing 2+
workouts.

Recommendation:
Create an early-warning rule for declining workout activity
and trigger personalized workout recommendations,
reminders, and re-engagement campaigns.

2. SUPPORT CONTACT
Customer-months with support contact had a churn rate of
{support_contact_rate:.2f}%, compared with
{no_support_contact_rate:.2f}% without support contact.

Recommendation:
Prioritize support cases when the customer is also showing
declining engagement. Track resolution time and post-support
retention.

3. ANNUAL RENEWAL RISK
Annual customers show a distinct retention risk around their
renewal period.

Recommendation:
Begin renewal engagement before the renewal month using
progress summaries, personalized recommendations and
targeted retention offers.

MEASUREMENT PLAN

Track:
- Monthly Churn Rate
- First-Month Engagement Rate
- Low-Engagement Churn Rate
- Support Contact Churn Rate
- Annual Renewal Rate
- Average Customer Tenure

Test retention actions using treatment and comparison groups
so leadership can measure whether interventions actually
reduce churn.

FINAL CONCLUSION
The strongest opportunity is to intervene before cancellation,
using declining engagement, customer-support contact and
renewal timing as actionable retention signals.
"""


print(recommendation)


# ===================================================
# 7. SAVE FINAL FILES
# ===================================================

kpi_summary.to_csv(
    "../../outputs/tables/kpi_summary.csv",
    index=False
)

final_churn_drivers.to_csv(
    "../../outputs/tables/final_churn_drivers.csv",
    index=False
)

annual_renewal_summary.to_csv(
    "../../outputs/tables/annual_renewal_summary.csv",
    index=False
)

cancel_reason_summary.to_csv(
    "../../outputs/tables/cancel_reason_summary.csv",
    index=False
)

monthly_churn.to_csv(
    "../../outputs/tables/monthly_churn_summary.csv",
    index=False
)


# Save recommendation as text file
with open(
    "../../docs/final_recommendation.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(recommendation)


print("\n------------------------------------")
print("STEP 20 COMPLETE")
print("------------------------------------")

print("Final files saved successfully.")

print("\nPython/Jupyter Phase: 20 / 20 COMPLETE ✅")

ANNUAL PLAN CHURN BY TENURE MONTH


,tenure_month_number,customer_months,churn_events,churn_rate_pct
0,0,1539,9,0.58
1,1,1524,12,0.79
2,2,1512,7,0.46
3,3,1385,13,0.94
4,4,1250,11,0.88
5,5,1138,11,0.97
6,6,1020,13,1.27
7,7,927,8,0.86
8,8,814,7,0.86
9,9,720,7,0.97



CANCEL REASON SUMMARY


,cancel_reason,churned_customers,cancel_reason_pct
2,not_using_enough,570,32.24
1,not_provided,382,21.61
5,too_expensive,322,18.21
0,found_alternative,175,9.90
3,other,162,9.16
4,technical_issues,157,8.88



FINAL TOP 3 CHURN DRIVERS


,driver,evidence,business_action,success_metric
0,Low Recent Engagement,0-1 previous-month workouts: 10.58% churn vs 4...,Trigger re-engagement campaigns when recent wo...,Monthly churn rate and re-engagement rate
1,Support Contact,Support contact: 11.66% churn vs 5.02% without...,Prioritize support cases for customers whose e...,Post-support churn and ticket resolution time
2,Annual Renewal Period,Annual renewal-period churn: 40.19%,Start renewal campaigns before annual customer...,Annual renewal rate and renewal-period churn



AI-POWERED SUBSCRIPTION CHURN & RETENTION INTELLIGENCE SYSTEM
FINAL BUSINESS RECOMMENDATION

BUSINESS PROBLEM
Leadership wants to understand why subscribers cancel and
which actions could improve retention.

KEY FINDINGS

1. LOW RECENT ENGAGEMENT
Customers completing 0-1 workouts in the previous month had
a churn rate of 10.58%, compared with
4.81% among customers completing 2+
workouts.

Recommendation:
Create an early-warning rule for declining workout activity
and trigger personalized workout recommendations,
reminders, and re-engagement campaigns.

2. SUPPORT CONTACT
Customer-months with support contact had a churn rate of
11.66%, compared with
5.02% without support contact.

Recommendation:
Prioritize support cases when the customer is also showing
declining engagement. Track resolution time and post-support
retention.

3. ANNUAL RENEWAL RISK
Annual customers show a distinct retention risk around their
renewal period.

Recommendation:
Begin renewal engagement before the renewal m